# 03. Model Benchmarking

We evaluate classical ML models on both our Regression target (NHANES: HbA1c) and Classification target (UCI: Readmission).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

# Note: In a real flow, we would load the preprocessed arrays from Notebook 2.
# For this notebook, we'll assume X_train_nh_prep, etc., are available in the workspace or passed via pickle.
# To make it self-contained for the prototype execution, we run the preprocessing pipeline here as well.

In [2]:
# Re-running preprocessing logic from Notebook 2
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

data_dir = '../data'
nhanes_files = ['DEMO_J.xpt', 'GHB_J.xpt', 'GLU_J.xpt', 'BIOPRO_J.xpt', 'BMX_J.xpt', 'DIQ_J.xpt', 'PAQ_J.xpt', 'SMQ_J.xpt']
nhanes_df = None
for file in nhanes_files:
    file_path = os.path.join(data_dir, file)
    if os.path.exists(file_path):
        df = pd.read_sas(file_path)
        nhanes_df = df if nhanes_df is None else pd.merge(nhanes_df, df, on='SEQN', how='outer')

nhanes_df = nhanes_df.dropna(subset=['LBXGH'])
y_nhanes = nhanes_df['LBXGH']
features = ['RIDAGEYR', 'RIAGENDR', 'BMXBMI', 'BMXWAIST', 'LBXGLU', 'PAQ605', 'SMQ020']
X_nhanes = nhanes_df[features].copy()
X_train_nh, X_test_nh, y_train_nh, y_test_nh = train_test_split(X_nhanes, y_nhanes, test_size=0.2, random_state=42)

num_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])

nhanes_preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, ['RIDAGEYR', 'BMXBMI', 'BMXWAIST', 'LBXGLU']),
    ('cat', cat_transformer, ['RIAGENDR', 'PAQ605', 'SMQ020'])
])

X_train_nh_prep = nhanes_preprocessor.fit_transform(X_train_nh)
X_test_nh_prep = nhanes_preprocessor.transform(X_test_nh)

### NHANES Regression Benchmarking

In [3]:
regressors = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),
    'Random Forest': RandomForestRegressor(n_estimators=50, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=50, random_state=42)
}

reg_results = []

# Impute missing targets if any (shouldn't be, but just in case)
y_train_nh = y_train_nh.fillna(y_train_nh.median())
y_test_nh = y_test_nh.fillna(y_train_nh.median())

for name, model in regressors.items():
    model.fit(X_train_nh_prep, y_train_nh)
    preds = model.predict(X_test_nh_prep)
    
    mae = mean_absolute_error(y_test_nh, preds)
    rmse = np.sqrt(mean_squared_error(y_test_nh, preds))
    r2 = r2_score(y_test_nh, preds)
    
    reg_results.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})

reg_df = pd.DataFrame(reg_results)
display(reg_df)

,Model,MAE,RMSE,R2
0,Linear Regression,0.441447,0.778120,0.380223
1,Ridge,0.441372,0.778062,0.380315
2,Lasso,0.601304,0.988612,-0.000446
3,ElasticNet,0.577451,0.945896,0.084141
4,Random Forest,0.473565,0.801170,0.342961
5,XGBoost,0.478947,0.829479,0.295707


In [4]:
# UCI Classification Preprocessing
uci_path = os.path.join(data_dir, 'diabetic_data.csv')
uci_df = pd.read_csv(uci_path, na_values='?')
uci_df['target'] = (uci_df['readmitted'] != 'NO').astype(int)
train_patients, test_patients = train_test_split(uci_df['patient_nbr'].unique(), test_size=0.2, random_state=42)
train_uci = uci_df[uci_df['patient_nbr'].isin(train_patients)].copy()
test_uci = uci_df[uci_df['patient_nbr'].isin(test_patients)].copy()
y_train_uci = train_uci['target']
y_test_uci = test_uci['target']

uci_features = ['age', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 
                'num_medications', 'number_outpatient', 'number_emergency', 
                'number_inpatient', 'number_diagnoses', 'diabetesMed']

uci_preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']),
    ('cat', cat_transformer, ['age', 'diabetesMed'])
])

X_train_uci_prep = uci_preprocessor.fit_transform(train_uci[uci_features])
X_test_uci_prep = uci_preprocessor.transform(test_uci[uci_features])

C:\Users\mishr\AppData\Local\Temp\ipykernel_20320\2839788185.py:3: DtypeWarning: Columns (0: payer_code) have mixed types. Specify dtype option on import or set low_memory=False.
  uci_df = pd.read_csv(uci_path, na_values='?')


### UCI Classification Benchmarking

In [5]:
classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=50, random_state=42)
}

clf_results = []

for name, model in classifiers.items():
    model.fit(X_train_uci_prep, y_train_uci)
    preds = model.predict(X_test_uci_prep)
    probs = model.predict_proba(X_test_uci_prep)[:, 1]
    
    acc = accuracy_score(y_test_uci, preds)
    prec = precision_score(y_test_uci, preds)
    rec = recall_score(y_test_uci, preds)
    f1 = f1_score(y_test_uci, preds)
    roc = roc_auc_score(y_test_uci, probs)
    prauc = average_precision_score(y_test_uci, probs)
    
    clf_results.append({'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'ROC-AUC': roc, 'PR-AUC': prauc})

clf_df = pd.DataFrame(clf_results)
display(clf_df)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Logistic Regression,0.619646,0.629525,0.388175,0.480232,0.654079,0.611254
1,Random Forest,0.583222,0.542854,0.502069,0.521665,0.607599,0.550079
2,XGBoost,0.623540,0.607212,0.476699,0.534098,0.663359,0.617203
